# Notebook 4 — Model Training & Evaluation

Trains two classifiers on the flow-level features:
1. **Random Forest** — strong baseline, interpretable feature importances
2. **XGBoost** — often best-in-class for tabular data

Class imbalance is handled with **SMOTE** (Synthetic Minority Over-sampling Technique).
The best model is saved to `models/best_model.pkl`.

In [ ]:
import sys
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

sys.path.insert(0, str(Path('.').resolve()))
from utils.feature_engineering import FEATURE_COLS

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_DIR   = Path('data')
MODELS_DIR = Path('models')
PLOTS_DIR  = DATA_DIR / 'plots'
MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
print('Feature columns:', FEATURE_COLS)

## 1. Load features

In [ ]:
features = pd.read_parquet(DATA_DIR / 'features.parquet')
print(f'Loaded {len(features):,} flows x {features.shape[1]} columns')
print('\nClass distribution (flows):')
print(features['label'].value_counts().to_string())

## 2. Prepare X and y

In [ ]:
X = features[FEATURE_COLS].copy()
y_raw = features['label'].copy()

# Encode string labels to integers for XGBoost compatibility
le = LabelEncoder()
y = le.fit_transform(y_raw)

print('Classes:', le.classes_)
print('Encoded:', list(range(len(le.classes_))))
print(f'\nX shape: {X.shape}')
print(f'y shape: {y.shape}')

# Save label encoder for inference
joblib.dump(le, MODELS_DIR / 'label_encoder.pkl')
print('\nLabel encoder saved.')

## 3. Train / test split (stratified 80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# Show class balance in train set
train_counts = pd.Series(y_train).map(dict(enumerate(le.classes_))).value_counts()
print('\nTrain class distribution:')
print(train_counts.to_string())

## 4. Scale features & apply SMOTE

SMOTE synthesises new samples for minority classes so all classes have equal representation.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Save scaler for inference
joblib.dump(scaler, MODELS_DIR / 'scaler.pkl')
print('Scaler saved.')

# SMOTE requires at least k_neighbors+1 samples per class
# Find the minimum class count and set k_neighbors accordingly
min_class_count = pd.Series(y_train).value_counts().min()
k_neighbors = min(5, min_class_count - 1)
print(f'Minimum class count: {min_class_count} → SMOTE k_neighbors={k_neighbors}')

smote = SMOTE(k_neighbors=k_neighbors, random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f'\nAfter SMOTE — Train: {len(X_train_res):,}')
resampled_counts = pd.Series(y_train_res).map(dict(enumerate(le.classes_))).value_counts()
print(resampled_counts.to_string())

## 5. Train Model 1 — Random Forest

In [ ]:
print('Training Random Forest...')
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=RANDOM_STATE
)
rf.fit(X_train_res, y_train_res)

y_pred_rf = rf.predict(X_test_scaled)
acc_rf    = accuracy_score(y_test, y_pred_rf)

print(f'\nRandom Forest Test Accuracy: {acc_rf:.4f} ({acc_rf*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(
    y_test, y_pred_rf,
    target_names=le.classes_,
    digits=4
))

## 6. Train Model 2 — XGBoost

In [ ]:
print('Training XGBoost...')
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    n_jobs=-1,
    random_state=RANDOM_STATE
)
xgb.fit(
    X_train_res, y_train_res,
    eval_set=[(X_test_scaled, y_test)],
    verbose=50
)

y_pred_xgb = xgb.predict(X_test_scaled)
acc_xgb    = accuracy_score(y_test, y_pred_xgb)

print(f'\nXGBoost Test Accuracy: {acc_xgb:.4f} ({acc_xgb*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(
    y_test, y_pred_xgb,
    target_names=le.classes_,
    digits=4
))

## 7. Compare models & select best

In [ ]:
print('=' * 45)
print(f'  Random Forest accuracy : {acc_rf:.4f}')
print(f'  XGBoost accuracy       : {acc_xgb:.4f}')
print('=' * 45)

if acc_xgb >= acc_rf:
    best_model = xgb
    best_name  = 'XGBoost'
    y_pred_best = y_pred_xgb
    best_acc    = acc_xgb
else:
    best_model = rf
    best_name  = 'Random Forest'
    y_pred_best = y_pred_rf
    best_acc    = acc_rf

print(f'\n✅ Best model: {best_name}  (accuracy = {best_acc:.4f})')

## 8. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_best)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # normalised

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title(f'{best_name} — Confusion Matrix (raw)', fontsize=12)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].tick_params(axis='x', rotation=30)

# Normalised
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title(f'{best_name} — Confusion Matrix (normalised)', fontsize=12)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '10_confusion_matrix.png', dpi=150)
plt.show()

## 9. Feature importance

In [ ]:
# Both RF and XGB expose feature_importances_
importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
importances = importances.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#2196F3' if v > importances.median() else '#90CAF9' for v in importances]
ax.barh(importances.index, importances.values, color=colors)
ax.set_title(f'{best_name} — Feature Importances', fontsize=13)
ax.set_xlabel('Importance score')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '11_feature_importance.png', dpi=150)
plt.show()

print('Top 5 most important features:')
print(importances.sort_values(ascending=False).head(5).to_string())

## 10. Save best model

In [ ]:
model_path = MODELS_DIR / 'best_model.pkl'
joblib.dump(best_model, model_path)
print(f'Best model ({best_name}) saved to {model_path}')

# Also save both models for comparison
joblib.dump(rf,  MODELS_DIR / 'random_forest.pkl')
joblib.dump(xgb, MODELS_DIR / 'xgboost.pkl')
print('Both models saved.')

# Save model metadata
import json
meta = {
    'best_model': best_name,
    'best_accuracy': float(best_acc),
    'rf_accuracy': float(acc_rf),
    'xgb_accuracy': float(acc_xgb),
    'classes': le.classes_.tolist(),
    'feature_cols': FEATURE_COLS,
    'n_flows_train': int(len(X_train_res)),
    'n_flows_test': int(len(X_test)),
}
with open(MODELS_DIR / 'model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Metadata saved.')
print(json.dumps(meta, indent=2))

## ✅ Notebook 4 complete
**Outputs:**
- `models/best_model.pkl` — best classifier
- `models/scaler.pkl` — fitted StandardScaler
- `models/label_encoder.pkl` — class name ↔ integer mapping
- `models/model_metadata.json` — accuracy & config summary

**Next:** Run `05_inference_pipeline.ipynb` to analyse your own Wireshark captures.